# Food-101 dish recognition + calorie estimation — CNN (GPU)

Stage 2 of the pipeline. Fine-tunes **EfficientNetB0** (ImageNet weights) on **Food-101**
(101 dishes, 101,000 images) and maps the predicted dish to calories with `calories.py`.

**Run on Colab:** Runtime → Change runtime type → GPU → Run all (Food-101 downloads via `tensorflow_datasets`, ~5 GB).
**Run on Kaggle:** enable GPU + Internet. Upload `calories.py`, `food101_classes.py` and `assets/food_calories.csv`
next to the notebook (or clone the repo in the first cell).

In [ ]:
# !git clone https://github.com/<your-username>/<this-repo>.git && %cd <this-repo>/05-food-recognition-calories
import numpy as np, tensorflow as tf, tensorflow_datasets as tfds, matplotlib.pyplot as plt
from calories import lookup
IMG, BATCH, SEED = 224, 64, 42
tf.keras.utils.set_random_seed(SEED)
(train_raw, test_raw), info = tfds.load('food101', split=['train', 'validation'], as_supervised=True, with_info=True)
CLASSES = info.features['label'].names
print(len(CLASSES), 'classes;', info.splits['train'].num_examples, 'train /', info.splits['validation'].num_examples, 'test')

In [ ]:
def prep(img, label):
    return tf.image.resize(img, (IMG, IMG)), label   # EfficientNet does its own normalisation
aug = tf.keras.Sequential([tf.keras.layers.RandomFlip('horizontal'), tf.keras.layers.RandomRotation(0.1),
                           tf.keras.layers.RandomZoom(0.15)])
train_ds = (train_raw.map(prep, num_parallel_calls=tf.data.AUTOTUNE).shuffle(4096, seed=SEED)
            .batch(BATCH).map(lambda x, y: (aug(x, training=True), y)).prefetch(tf.data.AUTOTUNE))
test_ds = test_raw.map(prep, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH).prefetch(tf.data.AUTOTUNE)

In [ ]:
base = tf.keras.applications.EfficientNetB0(include_top=False, weights='imagenet', input_shape=(IMG, IMG, 3))
base.trainable = False
inp = tf.keras.Input((IMG, IMG, 3))
x = base(inp, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
out = tf.keras.layers.Dense(len(CLASSES), activation='softmax')(x)
model = tf.keras.Model(inp, out)

# phase 1: train the new head
model.compile(tf.keras.optimizers.Adam(1e-3), 'sparse_categorical_crossentropy',
              metrics=['accuracy', tf.keras.metrics.SparseTopKCategoricalAccuracy(5, name='top5')])
h1 = model.fit(train_ds, validation_data=test_ds, epochs=3)

# phase 2: unfreeze the top of the backbone and fine-tune with a small learning rate
base.trainable = True
for layer in base.layers[:-60]:
    layer.trainable = False
model.compile(tf.keras.optimizers.Adam(1e-4), 'sparse_categorical_crossentropy',
              metrics=['accuracy', tf.keras.metrics.SparseTopKCategoricalAccuracy(5, name='top5')])
h2 = model.fit(train_ds, validation_data=test_ds, epochs=5,
               callbacks=[tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)])
loss, acc, top5 = model.evaluate(test_ds)
print(f'Food-101 test accuracy: top-1 {acc:.4f}, top-5 {top5:.4f}')
model.save('food101_efficientnetb0.keras')

In [ ]:
def recognise_and_estimate(image_path, top_k=3):
    img = tf.image.resize(tf.io.decode_image(tf.io.read_file(image_path), channels=3, expand_animations=False), (IMG, IMG))
    probs = model.predict(img[None], verbose=0)[0]
    out = []
    for i in probs.argsort()[::-1][:top_k]:
        cal = lookup(CLASSES[i])
        out.append({'dish': CLASSES[i], 'confidence': float(probs[i]),
                    'calories': f"{cal['kcal_per_serving']:.0f} kcal per {cal['serving']}" if cal else 'not in table'})
    return out

# try it on any food photo you upload:
# recognise_and_estimate('my_lunch.jpg')